In [1]:
import numpy as np

# Data

In [2]:
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

In [3]:
y = np.array([[0],
              [1],
              [1],
              [0]])

# Exercise

For the supplied XOR dataset, implement a simple feedforward neural network with one hidden layer with 2 units using sigmoid activation and 1 output unit with sigmoid activation. Train it using the MSE loss function and stochastic gradient descent (SGD), initializing the weights to small random values. Use a learning rate of 0.1, batch size of 2, and train for 10 epochs. After training, report the final weights and the predicted outputs for the XOR inputs.

## Solution

In [4]:
# Sigmoid activation and its derivative
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(z):
    a = sigmoid(z)
    return a * (1 - a)

# Reproducibility
np.random.seed(42)

# Network dimensions
input_size  = 2
hidden_size = 2
output_size = 1

# Weight initialisation (small random values)
W1 = np.random.randn(hidden_size, input_size)    # (2, 2)
b1 = np.zeros((hidden_size, 1))                        # (2, 1)
W2 = np.random.randn(output_size, hidden_size)   # (1, 2)
b2 = np.zeros((output_size, 1))                        # (1, 1)

# Hyper-parameters
lr         = 1
batch_size = 2
epochs     = 10000
n_samples  = X.shape[0]

Layers
$$
\mathbf{h}^{(1)} = \sigma \left( \mathbf{W}^{(1)} \mathbf{x} + \mathbf{b}^{(1)} \right)
$$
$$
\mathbf{h}^{(2)} = \sigma \left( \mathbf{W}^{(2)} \mathbf{h}^{(1)} + \mathbf{b}^{(2)} \right)
$$

In [5]:
# Hidden layer
def z1(x, W1, b1):
    return W1 @ x.reshape(2, 1)
def h1(x, W1, b1):
    return sigmoid(z1(x, W1, b1))

# Output layer
def z2(x, W2, b2):
    return W2 @ h1(x, W1, b1)
def h2(x, W2, b2):
    return sigmoid(z2(x, W2, b2))

Model
$$
\mathbf x \rightarrow \mathbf{z}^{(1)} \rightarrow \mathbf{h}^{(1)} \rightarrow \mathbf{z}^{(2)} \rightarrow \mathbf{h}^{(2)}
$$

In [6]:
# Model
def neural_net(x, W1, b1, W2, b2):
    z_1 = z1(x, W1, b1)
    h_1 = h1(x, W1, b1)
    z_2 = z2(h_1, W2, b2)
    h_2 = h2(h_1, W2, b2)
    return z_1, h_1, z_2, h_2

Loss
$$
L = \frac{1}{2} \left( {h}^{(2)} - {y} \right)^2
$$
$$
\frac{\partial L}{\partial {h}^{(2)}} = {h}^{(2)} - {y}
$$

In [7]:
def loss(y, h2):
    return 0.5 * ((h2 - y) ** 2)
def loss_deriv(y, h2):
    return np.array(h2 - y).reshape(1, 1)

Propagation errors
$$
\boldsymbol\delta^{(2)} = \frac{\partial L}{\partial \mathbf{z}^{(2)}} = \frac{\partial L}{\partial \mathbf{h}^{(2)}} \text{diag} \left( \sigma'(\mathbf{z}^{(2)}) \right) = \left( \mathbf{h}^{(2)} - \mathbf{y} \right)^\top \odot \text{diag} \left( \sigma'(\mathbf{z}^{(2)}) \right) = (y - h_2)\sigma'(z_2)
$$
$$
\boldsymbol\delta^{(1)} = \frac{\partial L}{\partial \mathbf{z}^{(1)}} = \boldsymbol\delta^{(2)} \left( \mathbf{W}^{(2)} \right)^\top \left( \sigma'(\mathbf{z}^{(1)}) \right)^\top = \left( \boldsymbol\delta^{(2)} \left( \mathbf{W}^{(2)} \right)^\top \right) \odot \left( \sigma'(\mathbf{z}^{(1)}) \right)^\top
$$

In [8]:
def delta_2(y, h2, z2):
    return (y - h2) * (sigmoid_deriv(z2)).T

def delta_1(delta_2, W2, z1):
    return (delta_2 @ W2) * sigmoid_deriv(z1).T

Gradients

In [9]:
# W_1
def d_L_W1(delta_1, x):
    return delta_1.T @ x.T
# b_1
def d_L_b1(delta_1):
    return delta_1.T

# W_2
def d_L_W2(delta_2, h1):
    return delta_2.T @ h1.T
# b_2
def d_L_b2(delta_2):
    return delta_2.T

Training

In [10]:
print(W1, b1, W2, b2)
print()
for n in range(epochs):
    # Create mini-batches
    batch_idx = np.random.permutation(n_samples)
    X_batch = X[batch_idx, :]
    y_batch = y[batch_idx, :]

    for i in range(X_batch.shape[0] // batch_size):
        dW2_batch = np.zeros(W2.shape)
        db2_batch = np.zeros(b2.shape)
        dW1_batch = np.zeros(W1.shape)
        db1_batch = np.zeros(b1.shape)
        # Forward pass
        for j in range(batch_size):
            sample_idx = i * batch_size + j
            z1_, h1_, z2_, h2_ = neural_net(X_batch[sample_idx], W1, b1, W2, b2)
        
            # Compute loss and gradients
            loss_value = loss(y_batch[j], h2_[0, 0])
            dL_dh2 = loss_deriv(y_batch[j], h2_[0, 0])
            
            # Backpropagation
            delta_2_ = delta_2(y_batch[j], h2_[0, 0], z2_)
            delta_1_ = delta_1(delta_2_, W2, z1_)

            # Gradients for weights and biases
            dW2_batch += d_L_W2(delta_2_, h1_)
            db2_batch += d_L_b2(delta_2_)
            dW1_batch += d_L_W1(delta_1_, X_batch[batch_idx[j]].reshape(2, 1))
            db1_batch += d_L_b1(delta_1_)

        # Update weights and biases
        W2 -= lr * dW2_batch
        b2 -= lr * db2_batch
        W1 -= lr * dW1_batch
        b1 -= lr * db1_batch


[[ 0.49671415 -0.1382643 ]
 [ 0.64768854  1.52302986]] [[0.]
 [0.]] [[-0.23415337 -0.23413696]] [[0.]]



In [11]:
W1, b1, W2, b2

(array([[1.79625774, 2.10767972],
        [1.83524968, 3.58494541]]),
 array([[3.38662481],
        [3.0711398 ]]),
 array([[-5.03306151, -5.76388135]]),
 array([[-7.26233696]]))

In [13]:
neural_net(X[3], W1, b1, W2, b2)

(array([[3.90393747],
        [5.42019509]]),
 array([[0.98023612],
        [0.99559322]]),
 array([[-10.6662561]]),
 array([[2.33181294e-05]]))